# Lab 1 — GPU Architecture Fundamentals: Microbenchmarks

**ROCm Certification Program — Level 1**

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Objectives</div>
<ul style='margin-bottom:0;'><li>Measure <b>memory bandwidth</b> with a streaming kernel</li><li>Measure <b>compute throughput</b> (GFLOPS) with a compute-heavy kernel</li><li>Collect hardware counters with <b>rocprofv3</b></li><li>Classify kernels as <b>memory-bound</b> or <b>compute-bound</b> using the roofline model</li></ul>
</div>


## 1. GPU Memory Hierarchy

Understanding where your data lives determines your kernel's performance ceiling.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    AMD Radeon GPU Memory Hierarchy                   │
│                                                                      │
│  ┌── Compute Unit (CU) ─────────────────────────────────────────┐    │
│  │                                                              │    │
│  │  Registers (VGPRs / SGPRs)                                   │    │
│  │    • Private to each work-item                               │    │
│  │    • Fastest storage on the GPU                              │    │
│  │    • Very limited capacity                                   │    │
│  │                                                              │    │
│  │  LDS (Local Data Share)                                      │    │
│  │    • 64 KB per Compute Unit                                  │    │
│  │    • Shared by all work-items in a workgroup                 │    │
│  │    • Software-managed                                        │    │
│  │    • Very low latency                                        │    │
│  │                                                              │    │
│  └──────────────────────────────────────────────────────────────┘    │
│                                                                      │
│  L1 Cache                                                            │
│    • 32 KB per Compute Unit (RDNA3)                                  │
│    • Low latency                                                     │
│                                                                      │
│  L2 Cache                                                            │
│    • Shared across the GPU                                           │
│    • 6 MB on this Radeon GPU                                         │
│                                                                      │
│  Infinity Cache (L3)                                                 │
│    • 96 MB on this Radeon GPU                                        │
│    • Reduces external memory traffic                                 │
│    • Improves effective bandwidth                                    │
│                                                                      │
│  GDDR6 Global Memory                                                 │
│    • Main GPU memory                                                 │
│    • Large capacity                                                  │
│    • High bandwidth                                                  │
│    • Higher latency than on-chip memories                            │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘


        Host (CPU) Memory
        Connected through PCIe

        PCIe Gen4 x16 : ~32 GB/s
        PCIe Gen5 x16 : ~64 GB/s

        Execution Model

        Wavefront size : 32 threads (this Radeon GPU)
        A wavefront is the basic execution unit scheduled on a Compute Unit.
```

<div style="background:#eef9f1; border-left:5px solid #2f9e6e; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1f7a52; font-size:1.05em; margin-bottom:8px;">💡 Key takeaway</div>
<p>Registers provide the highest bandwidth and lowest latency.</p> 
<p>LDS is the next fastest level and is shared within a workgroup.</p> 
<p>Accessing L1/L2/Infinity Cache is generally faster than accessing global memory</p>
<p>Optimizing GPU kernels often means maximizing data reuse in registers and LDS while minimizing global memory traffic.</p>
</div>

## 2. Memory Bandwidth Benchmark

This kernel streams data through HBM to measure achievable memory bandwidth. Each thread reads one float and writes one float — pure memory traffic, minimal compute.

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Your task</div>
<p>The <code>stream_triad</code> kernel computes <code>C[idx] = scalar*A[idx] + B[idx]</code> in several modes. Only the <i>memory access pattern</i> (how <code>idx</code> is computed) changes — the math is identical. For each mode:</p><ul><li>Run the kernel and confirm correctness (<b>Validation PASSED</b>)</li><li>Measure performance and compare modes</li></ul><p style='margin-bottom:0;'>If the absolute numbers look impossible, good catch — measuring GPU performance correctly is harder than it seems, and we revisit it later. For now, focus on <b>relative</b> behavior between patterns.</p>
</div>

In [ ]:
%%writefile mem_bandwidth.cpp
#include <hip/hip_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <chrono>

// ============================================================
// STREAM TRIAD MEMORY BANDWIDTH TEST
//
// Computes:
//
//     C[i] = scalar * A[i] + B[i]
//
// Each element performs:
//   - 2 global memory reads  (A and B)
//   - 1 global memory write  (C)
//
// This benchmark demonstrates:
//
//   1. Correct GPU indexing
//   2. Memory bandwidth
//   3. Coalesced vs non-coalesced access
//   4. Common GPU programming bugs
// ============================================================


// ============================================================
// MODE SELECTION
// ============================================================
//
// MODE_BASELINE
//   Ideal case.
//   Neighboring threads access neighboring memory.
//
// MODE_STRIDE
//   Poor memory coalescing.
//   Neighboring threads access distant locations.
//
// MODE_RANDOM
//   Pseudo-random access pattern.
//   Poor cache locality.
//
// MODE_INDEX_BUG
//   Incorrect indexing.
//   Ignores blockIdx.x so all blocks overwrite
//   the same memory region.
// ============================================================

#define MODE_BASELINE   0
#define MODE_STRIDE     1
#define MODE_RANDOM     2
#define MODE_INDEX_BUG  3

#ifndef MODE
#define MODE MODE_BASELINE
//#define MODE MODE_STRIDE
//#define MODE MODE_RANDOM
//#define MODE MODE_INDEX_BUG
#endif

static const char* spModes[] = 
{
    "MODE_BASELINE",
    "MODE_STRIDE",
    "MODE_RANDOM",
    "MODE_INDEX_BUG"
    
};

// CPU (host) memory
static float* h_A;
static float* h_B;
static float* h_C_ref;
static float* h_C_gpu;

// GPU (device) memory
static float* d_A;
static float* d_B;
static float* d_C;


// ============================================================
// GPU Kernel
// ============================================================

__global__ void stream_triad(
    const float* A,
    const float* B,
    float* C,
    float scalar,
    int N)
{
    int tid = threadIdx.x;

    // Global thread index
    int i = blockIdx.x * blockDim.x + tid;
    
    // BASELINE
    int idx = i;

#if MODE == MODE_STRIDE
    // Poor memory access pattern:
    // neighboring threads access distant elements
    idx = ((i * 65537) + 17) & (N - 1);
#elif MODE == MODE_RANDOM
     idx = ((i << 5) + i) & (N - 1); // almost random
#elif MODE == MODE_INDEX_BUG
    // WRONG:
    // ignores blockIdx.x
    // all blocks overwrite same region
    idx = tid;
#endif
    if (idx < N)
    {
        C[idx] = scalar * A[idx] + B[idx];
    }
}


// ============================================================
// Initialize host/device memory
// ============================================================

void init(int N, float scalar)
{
    hipError_t hipErr;

    size_t bytes = N * sizeof(float);

    // Allocate CPU memory
    h_A     = (float*)malloc(bytes);
    h_B     = (float*)malloc(bytes);
    h_C_ref = (float*)malloc(bytes);
    h_C_gpu = (float*)malloc(bytes);

    // Allocate GPU memory
    hipErr = hipMalloc(&d_A, bytes);
    hipErr = hipMalloc(&d_B, bytes);
    hipErr = hipMalloc(&d_C, bytes);

    // Initialize input vectors
    for (int i = 0; i < N; i++)
    {
        // (i % 1000) prevents floating-point errors when N is very lagre here
        h_A[i] = (i % 1000) * 0.5f;
        h_B[i] = (i % 1000) * 2.0f;
    }

    // Copy input vectors to GPU
    hipErr = hipMemcpy(d_A, h_A, bytes, hipMemcpyHostToDevice);
    hipErr = hipMemcpy(d_B, h_B, bytes, hipMemcpyHostToDevice);

    // CPU reference computation
    for (int i = 0; i < N; i++)
    {
        h_C_ref[i] = scalar * h_A[i] + h_B[i];
    }
}


// ============================================================
// Cleanup
// ============================================================

void cleanup()
{
    free(h_A);
    free(h_B);
    free(h_C_ref);
    free(h_C_gpu);

    hipError_t hipErr = hipFree(d_A);
    hipErr = hipFree(d_B);
    hipErr = hipFree(d_C);
}


// ============================================================
// Validation
// ============================================================

int check_kernel(int N, int block_size)
{
    size_t bytes = N * sizeof(float);

    int grid_size = (N + block_size - 1) / block_size;

    // Launch kernel
    stream_triad<<<grid_size, block_size>>>(
        d_A,
        d_B,
        d_C,
        3.0f,
        N);

    // Check launch errors
    hipError_t hipErr = hipGetLastError();

    if (hipErr != hipSuccess)
    {
        printf("Kernel launch failed: %s\n",
               hipGetErrorString(hipErr));
        return -1;
    }

    hipErr = hipDeviceSynchronize();

    // Copy result back to CPU
    hipErr = hipMemcpy(h_C_gpu, d_C, bytes, hipMemcpyDeviceToHost);

    const float epsilon = 1e-5f;

    float max_error = 0.0f;
    int max_error_index = 0;

    int mismatch_found = 0;

    for (int i = 0; i < N; i++)
    {
        float error = fabs(h_C_gpu[i] - h_C_ref[i]);

        if (error > max_error)
        {
            max_error = error;
            max_error_index = i;
        }

        if (error > epsilon)
        {
            printf("Validation FAILED\n");

            printf("First mismatch at index: %d\n", i);

            printf("CPU value: %f\n",
                   h_C_ref[i]);

            printf("GPU value: %f\n",
                   h_C_gpu[i]);

            printf("Error: %f\n", error);

            mismatch_found = 1;

            break;
        }
    }

    if (!mismatch_found)
    {
        printf("Validation PASSED\n");

        printf("Maximum error: %f at index %d\n",
               max_error,
               max_error_index);

        return 0;
    }

    return -1;
}


// ============================================================
// Benchmark
// ============================================================

double benchmark_kernel(
    int N,
    int block_size,
    int n_iters)
{
    int grid_size =
        (N + block_size - 1) / block_size;

    // Warmup
    for (int i = 0; i < 3; i++)
    {
        stream_triad<<<grid_size, block_size>>>(
            d_A,
            d_B,
            d_C,
            3.0f,
            N);
    }

    hipError_t hipErr = hipDeviceSynchronize();

    // Timed runs
    // NOTE:
    // This is a simplified benchmark for educational purposes.
    // Accurate GPU timing is more complicated and will be
    // discussed later.
        
    auto start =
        std::chrono::high_resolution_clock::now();

    for (int i = 0; i < n_iters; i++)
    {
        stream_triad<<<grid_size, block_size>>>(
            d_A,
            d_B,
            d_C,
            3.0f,
            N);
    }

    hipErr = hipDeviceSynchronize();
  
    auto end =
        std::chrono::high_resolution_clock::now();

    std::chrono::duration<double, std::milli>
        diff = end - start;

    double elapsed_ms = diff.count();

    // 2 reads + 1 write per element
    double total_bytes =
        (double)N *
        sizeof(float) *
        3 *
        n_iters;

    // Convert to GB/s
    double bw =
        ((total_bytes / 1e9) / elapsed_ms) * 1e3;

    printf(
        "  %-12s N=%10d  BW=%8.1f GB/s  time=%.4f ms\n",
        "stream_triad",
        N,
        bw,
        elapsed_ms / n_iters);

    return bw;
}


// ============================================================
// Main
// ============================================================

int main()
{
    printf("\n=== Memory Bandwidth Benchmark: MODE : %s ===\n\n", spModes[MODE]);

    int sizes[] =
    {
        1 << 18,
        1 << 20,
        1 << 22,
        1 << 24,
        1 << 26
    };

    int n_sizes = 5;
    int n_iters = 200;
    int block_size = 256;

    // Allocate maximum size once
    init(1 << 26, 3.0f);

    for (int s = 0; s < n_sizes; s++)
    {
        int N = sizes[s];

        printf(
            "Size: %d elements (%.1f MB)\n",
            N,
            (double)N * sizeof(float) / 1e6);

        check_kernel(N, block_size);

        benchmark_kernel(
            N,
            block_size,
            n_iters);

        printf("\n");
    }

    cleanup();

    return 0;
}


In [ ]:
!hipcc -O3 -o mem_bandwidth mem_bandwidth.cpp && ./mem_bandwidth

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">

<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">
Observations
</div>

<p>After running the benchmark, you should observe the following:</p>

<ul style="margin-bottom:0;">

<li>
The <b>baseline (coalesced)</b> kernel achieves the highest bandwidth because neighboring threads access neighboring memory locations.
</li>

<li>
Bandwidth is highest for medium-sized arrays that fit well within the GPU cache hierarchy. As the working set grows beyond the caches, performance becomes limited by global GDDR6 memory bandwidth.
</li>

<li>
<b>Strided access</b> significantly reduces bandwidth because neighboring threads access distant memory locations, resulting in poor memory coalescing.
</li>

<li>
<b>Random access</b> performs even worse for large arrays because cache locality is almost completely lost.
</li>

<li>
The <b>INDEX_BUG</b> kernel appears very fast, but fails validation because every workgroup repeatedly processes the same small memory region instead of the entire array.
</li>

<li>
Always verify <b>correctness before performance</b>. An incorrect kernel may report impressive bandwidth while doing much less useful work.
</li>

</ul>

</div>


<div style="background:#f3f0fb; border-left:5px solid #7c5cd6; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">

<div style="font-weight:700; color:#4c3a8c; font-size:1.05em; margin-bottom:8px;">
📝 Record your benchmark results
</div>

<p>
Run the <b>Triad</b> benchmark using the <b>MODE_BASELINE</b> configuration and record the measured bandwidth for each problem size.
</p>

<p>
Use the highest measured bandwidth as the reference (100%) and compute the relative performance for the remaining sizes.
</p>

</div>

| Elements | Size (MB) | Bandwidth (GB/s) | Relative to Best |
|----------:|----------:|-----------------:|-----------------:|
| 256K | | | |
| 1M   | | | |
| 4M   | | | |
| 16M  | | | |
| 64M  | | | |

> **Note:** The highest bandwidth does not necessarily occur for the largest array. GPU caches can significantly improve the effective bandwidth for medium-sized working sets. This benchmark measures **effective application bandwidth**, not the physical bandwidth of the GPU memory.

## 3. Compute Throughput Benchmark

This kernel performs many floating-point operations per byte loaded — pure compute with minimal memory traffic.

In [ ]:
%%writefile compute_bench.cpp
#include <hip/hip_runtime.h>
#include <stdio.h>
#include <chrono>

// Compute-heavy kernel: repeated FMA operations
__global__ void compute_kernel(float* out, int N, int n_ops) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) {
        float val = 1.0f;
        for (int j = 0; j < n_ops; j++) {
            val = val * 1.00001f + 0.00001f;  // FMA
        }
        out[i] = val;
    }
}

int main() {
    printf("\n=== Compute Throughput Benchmark ===\n\n");

    int N = 1 << 22;  // 4M threads
    size_t bytes = N * sizeof(float);
    float *d_out;
    hipError_t hipErr;

    hipErr = hipMalloc(&d_out, bytes);

    int block_sizes[] = {64, 128, 256, 512, 1024};
    int n_blocks = 5;
    int n_ops = 1000;  // FMA ops per thread
    int n_iters = 20;

    for (int b = 0; b < n_blocks; b++) {
        int block = block_sizes[b];
        int grid = (N + block - 1) / block;

        // Warmup
        compute_kernel<<<grid, block>>>(d_out, N, n_ops);
        hipErr = hipDeviceSynchronize();

        // Timed
        auto start = std::chrono::high_resolution_clock::now();
        for (int i = 0; i < n_iters; i++)
            compute_kernel<<<grid, block>>>(d_out, N, n_ops);
        hipErr = hipDeviceSynchronize();
        auto end = std::chrono::high_resolution_clock::now();
        double elapsed = std::chrono::duration<double>(end - start).count();

        // 2 FLOP per FMA (multiply + add)
        double total_flops = 2.0 * n_ops * N * n_iters;
        double gflops = total_flops / elapsed / 1e9;

        printf("  Block=%4d  Grid=%6d  GFLOPS=%8.1f  time/iter=%.4f ms\n",
               block, grid, gflops, elapsed / n_iters * 1000);
    }

    hipErr = hipFree(d_out);
    return 0;
}

In [ ]:
!hipcc -O3 -o compute_bench compute_bench.cpp && ./compute_bench

## Roofline Classification

The Roofline Model helps determine whether a kernel is primarily limited by **memory bandwidth** or by **compute throughput**.

```
Performance
(GFLOPS)
            │                         Peak Compute
            │──────────────────────────────────────────────
            │                      /
            │                    /
            │                  /      COMPUTE-BOUND
            │                /        Increase arithmetic
            │              /          efficiency
            │            /
            │          /
            │        /
            │      /          MEMORY-BOUND
            │    /            Improve memory access
            │  /              and data reuse
            │/
            └──────────────────────────────────────────────
                  Arithmetic Intensity (FLOP/Byte)
```
### Arithmetic Intensity

Arithmetic Intensity (AI) measures how much computation is performed for each byte of memory traffic.

```
AI = Total FLOPs / Total Bytes Transferred
```

- Low AI → Memory-bound
- High AI → Compute-bound

Higher arithmetic intensity means that more computation is performed for each byte loaded from or stored to memory.

---

### Key Takeaways

- Low arithmetic intensity usually indicates a **memory-bound** kernel.
- High arithmetic intensity usually indicates a **compute-bound** kernel.
- Memory-bound kernels benefit most from improving memory access patterns and data reuse.
- Compute-bound kernels benefit most from improving arithmetic efficiency, occupancy, and instruction throughput.
- The Roofline Model helps identify which optimization strategy is most likely to improve performance.

## 5. Profile with rocprofv3

Collect hardware counters to confirm the roofline classification.

```

Observe that the profiler reports:

- kernel execution time
- kernel launch count
- GPU activity

The appearance of the report may differ between ROCm versions, but the information is the same.
```

In [ ]:
# Profile the memory bandwidth kernel
!rocprofv3 --stats --kernel-trace -T -o ./profiler/mem_bandwidth -- ./mem_bandwidth 2>&1 | tail -20

In [ ]:
# Profile the compute kernel
!rocprofv3 --stats --kernel-trace -T -o ./profiler/compute_bench -- ./compute_bench 2>&1 | tail -20

## 6. Results Visualization

In [ ]:
# Results Visualization

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

# ============================================================
# Memory Bandwidth Results (GB/s)
# ============================================================

sizes_mb = [1, 4, 16, 64, 256]

baseline = [605.6, 1089.2, 1950.4, 730.8, 741.1]
stride   = [415.9, 152.6, 118.4, 77.7, 67.4]
random   = [153.9, 145.0, 117.7, 23.9, 24.4]

# ============================================================
# Create Figure
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

# Baseline
ax.plot(
    sizes_mb,
    baseline,
    "o-",
    linewidth=3,
    markersize=8,
    label="Baseline (Coalesced)"
)

# Strided
ax.plot(
    sizes_mb,
    stride,
    "s-",
    linewidth=2,
    markersize=7,
    label="Strided"
)

# Random
ax.plot(
    sizes_mb,
    random,
    "^--",
    linewidth=2,
    markersize=7,
    label="Random"
)

# Highlight peak
peak_idx = baseline.index(max(baseline))

ax.scatter(
    sizes_mb[peak_idx],
    baseline[peak_idx],
    color="red",
    s=120,
    zorder=5
)

ax.annotate(
    "Peak(Cache-friendly\nworking set)",
    xy=(sizes_mb[peak_idx], baseline[peak_idx]),
    xytext=(35, -5),
    textcoords="offset points",
    fontsize=10,
    arrowprops=dict(arrowstyle="->", lw=1.2)
)

# Shade large working sets
ax.axvspan(
    32,
    300,
    color="gray",
    alpha=0.08
)

ax.text(
    70,
    1750,
    "Mostly Global Memory",
    fontsize=10,
    color="gray"
)

# Formatting
ax.set_xscale("log")

ax.set_xlabel("Buffer Size (MB)", fontsize=12)
ax.set_ylabel("Effective Bandwidth (GB/s)", fontsize=12)
ax.set_title(
    "Memory Bandwidth vs Buffer Size",
    fontsize=14,
    fontweight="bold"
)

ax.grid(True, alpha=0.3)

ax.legend()

# Cleaner appearance
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    "lab1b_results.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("Plot saved to lab1b_results.png")

## Key Takeaways

### 1. Memory Access Patterns Matter

Modern GPUs can process enormous amounts of data, but only if that data
is accessed efficiently.

Sequential and coalesced memory accesses allow the hardware to fully utilize
the available HBM bandwidth. Poor access patterns can dramatically reduce
performance even when the amount of computation remains unchanged.

In this lab, we observed that changing only the memory access pattern could
significantly impact throughput.

---

### 2. More Compute Is Not Always Faster

A GPU contains thousands of arithmetic units (ALUs), but their performance
depends on keeping them supplied with work and data.

By increasing the amount of computation per memory access, we shifted the
kernel from a memory-bound workload toward a compute-bound workload.

This demonstrates an important optimization principle:

```text
Performance is often limited by either:
  • memory bandwidth
  • compute throughput
```

Improving the wrong resource may have little effect on overall performance.

---

### 3. Occupancy Influences Throughput

Thread block size affects how many wavefronts can execute concurrently on
each Compute Unit (CU).

Too few threads may leave hardware resources idle, while too many threads
can increase register and shared-memory pressure.

Finding an efficient launch configuration is often an important part of GPU
performance tuning.

---

### 4. Every GPU Has a Ceiling

Memory bandwidth and floating-point throughput are finite resources.

As problem size grows, performance eventually approaches one of these
hardware limits.

Once a bottleneck is reached, further optimization must target that
specific resource rather than the application as a whole.

---

### 5. The Roofline Model Helps Identify Bottlenecks

The Roofline Model combines:

- Compute Performance (GFLOPS)
- Memory Bandwidth (GB/s)
- Arithmetic Intensity (FLOPs per byte)

to determine whether a kernel is limited primarily by memory or by compute.

Understanding where a kernel lies on the roofline helps engineers focus
their optimization effort where it will have the greatest impact.

<div style="
    background:#eef8ee;
    border-left:5px solid #68a36d;
    padding:12px;
    border-radius:6px;
    margin:15px 0;
">

<b>Final Thought</b><br><br>

Many GPU optimizations can be reduced to a simple question:

<b>Am I waiting for data, or am I waiting for computation?</b>

The experiments in this lab demonstrate how memory access patterns,
arithmetic intensity, and occupancy determine the answer.

</div>

---

## Summary

| Task | Status |
|------|--------|
| Memory bandwidth benchmark runs across 5 sizes | ☐ |
| Compute throughput measured across block sizes | ☐ |
| rocprofv3 profiles collected | ☐ |
| Kernels classified as memory/compute-bound | ☐ |
| Results plotted | ☐ |


**Next:** Module 2 — Kernel Tuning